# LangChain Postgres Chat Message History with Ollama

This notebook demonstrates how to build a stateful conversational LLM application using:
- **LangChain** as the orchestration framework
- **Ollama (Llama 3)** running locally as the LLM
- **PostgreSQL** to store and manage chat session histories persistently

### Prerequisites
Ensure you have a running PostgreSQL database and Ollama instance installed locally.
```bash
pip install langchain langchain-ollama langchain-postgres sqlalchemy psycopg
```

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import uuid
from langchain_postgres import PostgresChatMessageHistory
from sqlalchemy import create_engine


## 1. Initialize the Chat Model
We use `ChatOllama` to connect to a locally running Llama 3 model.

In [2]:
model = ChatOllama(model="llama3:8b-instruct-q4_K_M")

## 2. Define the Prompt Template
We construct a conversational prompt template containing:
- A system instruction
- A `MessagesPlaceholder` called `History` to inject historical messages
- A human message input placeholder

In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="History"),
        ("human", "{input}")
    ]
)

In [4]:
chain = prompt | model

## 3. Configure PostgreSQL Database Connection
We configure SQLAlchemy to connect to our Postgres database and create the required schema tables to store history.

In [5]:
DB_URL = (
    "postgresql+psycopg://"
    "chatuser:1234"
    "@localhost:5432/chatmodel"
)

In [6]:
engine = create_engine(DB_URL)

In [7]:
table_name = "chat_history"
conn = engine.raw_connection()
PostgresChatMessageHistory.create_tables(conn, table_name)

## 4. Define Session History Provider
We create a helper function `get_session(session_id)` which returns a `PostgresChatMessageHistory` object tied to a specific session ID.

In [8]:
session_id = str(uuid.uuid4())

In [9]:
def get_session(session_id):
    return PostgresChatMessageHistory(
        table_name,
        session_id,
        sync_connection=conn
    )

In [10]:
test_his = get_session(session_id)

In [11]:
print(test_his.messages)

[]


## 5. Invoking the Chain Manually
First, we run the chain by manually retrieving and feeding the history list. Notice that the chat history is not automatically saved back to the database in this raw execution.

In [12]:
chain.invoke({"input":"my name is john i am from UK and i am 15 years old","History":test_his.messages})

AIMessage(content="Nice to meet you, John! I'm happy to be your helpful assistant. It's great that you're from the UK and only 15 years young!\n\nSo, what brings you here today? Do you have a specific question or topic you'd like to chat about? Or maybe you just want to introduce yourself and see how our conversation goes?", additional_kwargs={}, response_metadata={'model': 'llama3:8b-instruct-q4_K_M', 'created_at': '2026-06-05T16:38:33.2560655Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20498445300, 'load_duration': 16254843400, 'prompt_eval_count': 36, 'prompt_eval_duration': 289857000, 'eval_count': 71, 'eval_duration': 3926977000, 'logprobs': None, 'model_name': 'llama3:8b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--019e98a6-4004-7110-9ddc-0a8dd781e2f1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 71, 'total_tokens': 107})

In [13]:
print(test_his.messages)

[]


## 6. Automating History with `RunnableWithMessageHistory`
To automatically save and retrieve chat messages, we wrap our chain with `RunnableWithMessageHistory`.

> **Important Configuration Notes:**
> - **`input_messages_key`**: Specifies the dictionary key holding the user prompt (`"input"`).
> - **`history_messages_key`**: Specifies the variable name in the prompt template's `MessagesPlaceholder` (`"History"`).

In [14]:
agent_with_history = RunnableWithMessageHistory(
    chain,
    get_session,
    input_messages_key="input",
    history_messages_key="History"

)

d:\Rag-Projects\Langchain\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
config = {
    "configurable": {
        "session_id": session_id
    }
}

## 7. Testing stateful conversational conversation
Now we test the agent over consecutive interactions, verifying that the model remembers context across calls (stored and retrieved from PostgreSQL).

In [16]:
agent_with_history.invoke({"input":"my name is john i am from UK and i am 15 years old"},config=config)

AIMessage(content="Nice to meet you, John! I'm happy to be your helpful assistant. It's great that you're 15 and from the UK - that's awesome!\n\nSo, what brings you here today? Do you have any questions or topics you'd like to discuss? Are you looking for help with something specific or just want to chat?", additional_kwargs={}, response_metadata={'model': 'llama3:8b-instruct-q4_K_M', 'created_at': '2026-06-05T16:39:04.1320754Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5852488900, 'load_duration': 518205500, 'prompt_eval_count': 36, 'prompt_eval_duration': 430533000, 'eval_count': 69, 'eval_duration': 4142160000, 'logprobs': None, 'model_name': 'llama3:8b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--019e98a6-f1dc-7e21-9628-49a96000b60e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 69, 'total_tokens': 105})

In [16]:
agent_with_history.invoke({"input":"what is my name? and where am from and my age"},config=config)

AIMessage(content='Easy one!\n\nYour name is John.\n\nYou are from the United Kingdom (UK).\n\nAnd you are 15 years old.', additional_kwargs={}, response_metadata={'model': 'llama3:8b-instruct-q4_K_M', 'created_at': '2026-06-05T14:53:24.935029Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2703954600, 'load_duration': 824153900, 'prompt_eval_count': 144, 'prompt_eval_duration': 555733000, 'eval_count': 26, 'eval_duration': 1302632000, 'logprobs': None, 'model_name': 'llama3:8b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--019e9846-43b4-7890-9300-3be067f16d69-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_tokens': 26, 'total_tokens': 170})

## 8. Interactive Chat Loop
You can run the cell below to start an interactive chat session with the assistant. Type `exit` or `quit` to end the conversation. The chat history will be automatically stored in the PostgreSQL database under the configured `session_id`.

In [17]:
print("Start chatting with the agent! Type 'exit' or 'quit' to end the session.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("\nSession ended. Goodbye!")
        break
    
    response = agent_with_history.invoke(
        {"input": user_input},
        config=config
    )
    print(f"Agent: {response.content}\n")


Start chatting with the agent! Type 'exit' or 'quit' to end the session.

Agent: Nice to meet you, Ram! I'm happy to be your helpful assistant. You're from the UK and 16 years old, that's cool!

So, what brings you here today? Do you have any questions or topics you'd like to discuss? Are you looking for help with something specific or just want to chat?

By the way, I noticed earlier you mentioned your name was John and you were 15. Just to clarify, is Ram actually your correct name, and you're 16 years old now?

Agent: I think there might be some confusion! Earlier, you told me that your name is John and you are 15 years old. Then, you corrected yourself and said your name is Ram and you are 16 years old.

So, to clarify, I'm going to ask again: What is your correct name (John or Ram?), and how old are you now?


Session ended. Goodbye!
